# Simple GNN Test with Augmented Data

Now that the model is trained and tested on one graph, we rotate the same image, generate graph, and see if the model can still handle it.

In [ ]:
%load_ext autoreload
%autoreload 1
%aimport image_processing_tools

## Load and rotate data

In [ ]:
from image_processing_tools.util.load_files import find_files_by_pattern
from image_processing_tools.image_class.image_container import ImageContainer
from pathlib import Path
import matplotlib
# matplotlib.use('Agg') # Critical: prevents plotting to screen/RAM

search_path = ("~/A8/Data_Roan/260123_Cocu_Phalloidin/Cocu_Cet112_Ca922_Phalloidin_low_01_CY5, DAPI",
               "~/A8/Data_Roan/260228_She3_Mutants_DAPI",)
file_pattern = ("CROP_*",
                "MAX_*",
                "SHARPEST*",
                "MAX_CET145*")

config = {
    "preprocessing": {
        "resize_image": False,
        "max_dim": 1080,
        "outlier_percentile": 0.35,
        "quantization": "8bit"
    }
}

# Find the files. The 'files' variable will hold the list of Path objects.
files = find_files_by_pattern(search_path[0], file_pattern[0], verbose=True)

In [ ]:
from image_processing_tools.dapi_tracing.gnn_data import augment_rotate

crop_img = ImageContainer(files[0],config).merge()

rotated_imgs = augment_rotate(crop_img)

In [ ]:
from image_processing_tools.image_class.image_filters import ImageJProcessor
import matplotlib.pyplot as plt

def run_filters(img, show_plot=True):
    pipeline_seg = ImageJProcessor(img)
    pipeline_seg.reset()
    pipeline_seg.fft_bandpass_filter()
    pipeline_seg.enhance_contrast_clahe(slope=1.5)
    pipeline_seg.imagej_rolling_ball(radius=60)
    pipeline_seg.gaussian_blur(sigma=3)
    seg_img = pipeline_seg.return_image()

    pipeline_intens = ImageJProcessor(img)
    pipeline_intens.reset()
    pipeline_intens.fft_bandpass_filter()
    pipeline_intens.enhance_contrast_clahe(slope=1)
    pipeline_intens.gaussian_blur(sigma=5)
    intens_img = pipeline_intens.return_image()

    if show_plot:
        fig, axes = plt.subplots(1,2,figsize=(8,4))
        axes[0].imshow(seg_img, cmap='gray'); axes[0].set_title('Image for Segmentation')
        axes[1].imshow(intens_img, cmap='gray'); axes[1].set_title('Image for Intensity')

        for ax in axes: ax.axis('off')
        # plt.axis('off')
        plt.tight_layout()
        plt.show()
        
    return seg_img, intens_img

processed_imgs = [run_filters(img) for img in rotated_imgs]

In [ ]:
len(processed_imgs[0])

## Extract features

In [ ]:
from image_processing_tools.dapi_tracing.connectivity_score import detect_nuclei, filter_nuclei, extract_graph

def extract_graph_pipeline(seg_img,intens_img):
    labels, binary_mask_filled = detect_nuclei(seg_image=seg_img, use_watershed=False)
    valid_nuclei_data,_,_ = filter_nuclei(labels)
    nuclei_df, paths_df, edge_index = extract_graph(valid_nuclei_data, intens_img, binary_mask_filled)
    
    return nuclei_df, paths_df, edge_index

nuclei_df_list, paths_df_list, edge_index_list = [],[],[]

for i in range(len(processed_imgs)):
    seg_img, intens_img = processed_imgs[i]
    nuclei_df, paths_df, edge_index = extract_graph_pipeline(seg_img,intens_img)
    nuclei_df_list.append(nuclei_df)
    paths_df_list.append(paths_df)
    edge_index_list.append(edge_index)

In [ ]:
import pandas as pd

pd.DataFrame(edge_index_list[3])

In [ ]:
edge_labels_list = []
edge_labels_list.append([1,0,0,0,1,0,0,1,0,1])
edge_labels_list.append([0,0,1,0,1,0,0,0,1,1])
edge_labels_list.append([1,0,0,0,1,0,0,1,0,1])
edge_labels_list.append([1,1,0,0,0,0,1,1,0,0])

## Create list of pyg data objects

In [ ]:
from image_processing_tools.dapi_tracing.gnn_data import create_pyg_data

pyg_data_list = create_pyg_data(edge_indices = edge_index_list, 
                                nuclei_features_list = nuclei_df_list, 
                                path_features_list = paths_df_list, 
                                edge_labels_list = edge_labels_list)

## Run 4-fold cross validation

In [ ]:
from image_processing_tools.dapi_tracing.gnn_data import n_fold_validation

model_params = {'hidden_channels': 32,
                'dropout_p': 0.2}

results = n_fold_validation(dataset = pyg_data_list,
                            num_folds = 2,
                            max_epochs = 300,
                            batch_size = 2,
                            learning_rate = 0.1,
                            model_params=model_params,
                            log_dir_append='dynamic_edge_attr_relative_features_dropout_2fold_2')